## Plots of ECH Adaptation


In [1]:
from pathlib import Path

import altair as alt
import pandas as pd
import polars as pl


monthYear = "%b %Y"

DATA_DIR = Path("../../data/measurements")

df_activation = pl.read_parquet(DATA_DIR / "ech_adaptation.parquet")
df_deactivation = pl.read_parquet(DATA_DIR / "ech_deactivation.parquet")

df = pl.concat(
    [
        # (
        #     df_activation.pivot(index="date", values="value", on="metric")
        #     .with_columns((pl.col("total_ech") - pl.col("cloudflare_ech")).alias("non_cloudflare_ech"))
        #     .unpivot(
        #         index="date",
        #         on=[
        #             "total_ech",
        #             "total_tested_domains",
        #             "distinct_ech_configs",
        #             "https_records",
        #             "cloudflare_ech",
        #             "non_cloudflare_ech",
        #         ],
        #     )
        # ),
        df_activation.select(
            pl.col("date"),
            pl.col("metric").alias("variable"),
            pl.col("value"),
        ),
        df_deactivation.select(
            pl.col("current_test_date").alias("date"),
            pl.lit("ech_deactivation").alias("variable"),
            pl.col("deactivated_ech_count").alias("value"),
        ),
    ]
).filter(pl.col("value") > 0)

In [5]:
FONT_CHOICE = 'monospace'

final_chart = (
    (line_chart + cf_rule + cf_start_label + cf_end_label)
    .properties(
        width=450,
        height=180,
    )
    .configure_legend(
        orient="none",
        direction="horizontal",
        legendX=0,
        legendY=225,
        labelFont=FONT_CHOICE,
        titleFont=FONT_CHOICE,
        labelFontSize=12,
        titleFontSize=12
    )
    .configure_axis(
        labelFont=FONT_CHOICE,
        titleFont=FONT_CHOICE,
        labelFontSize=12,
        titleFontSize=12,
        grid=True,
        gridColor="#E8E8E8",
        domainColor="#000000",
        tickColor="#000000"
    )
    .configure_text(
        font=FONT_CHOICE,
        fontSize=11,
        color="#333333"
    )
    .configure_view(
        strokeWidth=0
    )
)

final_chart.save("../../plots/ech_adoption.pdf", engine="vl-convert")
final_chart.show()

alt.LayerChart(...)